In [ ]:
# get the data set of 32x32 RGB images from the CIFAR-10 dataset
from torchvision import datasets

# define where the data will be stored
data_path = '../../data/data-unversioned/p1ch7/'

# download the training data
cifar10 = datasets.CIFAR10(data_path, train = True, download = True)

# download the validation data
cifar10_val = datasets.CIFAR10(data_path, train = False, download = True)

Other popular "standard" data sets that are baked in PyTorch are
* MNIST
* Fashion-MNIST
* CIFAR-100
* SVHN
* Coco
* Omniglot

In [ ]:
# mro = method-resolution order: 
# the mro defines which order of super classes
# the object (cifar10, here) will search for 
# when any method is called. This is to prevent
# any issues relating to the same method name 
# being used in multiple super classes of this 
# object.
#
# This is also to highlight that this object, ultimately,
# is a subclass of the torch.utils.data.dataset.Dataset 
# class, meaning that there are plenty of other standard
# datasets to explore, we just happened to go down the
# vision models rabbit hole :)
type(cifar10).__mro__

In [ ]:
# The main methods for torch.utils.data.dataset.Dataset
# are the dunder methods __len__ and __getitem__
# The former lets us know how many data points we
# have via the syntactic surgar len(Object) while the latter 
# allows us to use the syntactic suger Object[i] to get
# a specific data point.
len(cifar10)

In [ ]:
# define the class names as a list (corresponds to class)
class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

img, label = cifar10[99]
img, label, class_names[label]

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(img)
plt.show()

In [ ]:
# import transforms (set of functions that can be passed
# to the dataset object and will be acted upon each data
# point with prior to any __getitem__ call)
from torchvision import transforms
dir(transforms)

In [ ]:
# to ToTensor transform that can handle numpy arrays 
# and PIL image formats (at the time of the book being written)

# instantiate the transform object (just a function)
to_tensor = transforms.ToTensor()

img_t = to_tensor(img) # note that we have only acted upon the image, not the label which was unpacked separately
img_t.shape

In [ ]:
# send the transform to the entire data set
tensor_cifar10 = datasets.CIFAR10(data_path, 
                                  train=True, 
                                  download=False, # if true, downloads the dataset if it is not in the location path already
                                  transform = transforms.ToTensor() # aparently knows to only touch the PIL files
                                  )

In [ ]:
img_t, _ = tensor_cifar10[99]
type(img_t)

In [ ]:
img_t.shape, img_t.dtype # ToTensor automatically scales the 0 to 255 int data type to the standar 0 to 1 float32

In [ ]:
img_t.min(), img_t.max()

In [ ]:
plt.imshow(img_t.permute(1,2,0)) # imshow expects HxWxC
plt.show()

In [ ]:
# The normalization of ToTensor is not quite what we want,
# so we will use transforms.Compose to get what we want. 
# In what follows, we can write out explicitly what will 
# happen when we call standard functions from transforms

# since the data is small in terms of memory, grab all data
import torch
# note that we are invoking a new dimension at the end
# to store the entire batch (rather than in front). So,
# we will probably need to permute dimensions later
imgs = torch.stack([img_t for img_t, _ in tensor_cifar10], dim=3)
imgs.shape 

In [ ]:
# view is a method to reshape the tensor
# here we are specifying that we want a
# tensor of shape (3, whatever is the length of the final dimension)
# it will start left to right, so since we already have a 3x32x32x50000
# tensor, it will leave the first dimension alone and basically 
# stack the remaining data into a vector
imgs.view(3,-1).mean(dim=-1) # take mean over last dimesion, i.e. all pixel intensities of each channel separately

In [ ]:
imgs.view(3,-1).std(dim=-1)

In [ ]:
# Normalize transform expects means and variances

means = imgs.view(3,-1).mean(dim=-1)
stds = imgs.view(3,-1).std(dim=-1)

transforms.Normalize(means, stds)

In [ ]:
# Now apply to the dataset
transformed_cifar10 = datasets.CIFAR10(
    data_path,
    train = True,
    download = False,
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(means, stds)
    ])
)

In [ ]:
img_t, _ = transformed_cifar10[99]

plt.imshow(img_t.permute(1,2,0)) # matplotlib will render data outside 0,1 range as black
plt.show()